# Пайплайн аугментации и классификации (Google Colab)

Ноутбук последовательно запускает все этапы аугментации несбалансированного датасета
и оценивает результат через baseline-классификаторы.

**Этапы аугментации:**
1. LLM-генерация (< 15 → 15)
2. Парафраз через LLM (< 35 → 35)
3. Обратный перевод (< 50 → 50)

**Классификация (baseline):**
- Linear SVM
- Logistic Regression
- Gaussian Naive Bayes

> **Среда:** Google Colab (GPU runtime рекомендуется)

## 0. Подготовка окружения

Монтируем Google Drive, ставим зависимости, настраиваем пути.

In [12]:
# Монтируем Google Drive — тут лежат данные и код
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import sys
from pathlib import Path

# Корень проекта на Google Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/VKR/code")

# Добавляем корень в sys.path, чтобы работали импорты вида from src.utils...
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Переходим в корень проекта — чтобы относительные пути в модулях работали корректно
%cd {PROJECT_ROOT}

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"Папка данных:   {PROJECT_ROOT / 'Data'}")

/content/drive/MyDrive/VKR/code
Корень проекта: /content/drive/MyDrive/VKR/code
Папка данных:   /content/drive/MyDrive/VKR/code/Data


In [14]:
ls /content/drive/MyDrive/VKR/code/Data

data_after_eda.csv  data.json


In [15]:
# Ставим зависимости из requirements.txt
!pip install -q -r {PROJECT_ROOT}/requirements.txt

In [16]:
# for i, text in enumerate(df[df["label"] == "Блок заместителя генерального директора по строительству"]["text"], 1):
#     print(f"--- Письмо {i} ---")
#     print(text)
#     print()

## 1. Смотрим на исходные данные

In [17]:
import pandas as pd
from src.utils.data_loader import load_dataset, get_class_distribution, LABEL_COL

# Загружаем исходный датасет (после EDA)
df_original = load_dataset(stage=0)

print(f"\nВсего записей: {len(df_original)}")
print(f"Классов: {df_original[LABEL_COL].nunique()}")
print(f"\nРаспределение по классам:")
dist = get_class_distribution(df_original)
for name, count in dist.items():
    print(f"  {name}: {count}")

[Данные] Найден чекпоинт этапа 0: data_after_eda.csv (1755 записей)

Всего записей: 1755
Классов: 36

Распределение по классам:
  Блок технического директора: 246
  Блок директора по мощностям: 241
  Блок директора по строительству: 165
  Управление по проектным работам: 135
  Блок заместителя генерального директора по безопасности: 123
  Генеральный директор: 102
  Проект "Нефтяные краюшки": 79
  Блок деректора по газу: 71
  Блок заместителя генерального директора по закупкам: 67
  Блок заместителя генерального директора по организационным вопросам: 56
  Проект сервиса скважин: 45
  Блок директора по проектированию: 43
  Проект "Новая нефть": 38
  Проект "Северная деревня": 36
  Блок операционного директора: 33
  Блок директора по газовым проектам: 31
  Блок заместителя генерального директора по защите: 26
  Блок финансового директора: 25
  Блок директора по портфелю: 24
  Управление землеустроительных работ: 23
  Проект "Трубопроводный транспорт Главного НГКМ": 20
  Проект "Восточный

## 2. Этап 1: LLM-генерация (< 15 -> 15)

Классы с менее чем 15 примерами дополняются новыми текстами,
сгенерированными через LLM (модель из JSON-конфига).

In [18]:
# Путь до конфига модели 
# CONFIG_PATH = str(PROJECT_ROOT / "configs" / "model_qwen.json")
# урезанная
# CONFIG_PATH = str(PROJECT_ROOT / "configs" / "model_qwen_3b.json")
# квантизированная
CONFIG_PATH = str(PROJECT_ROOT / "configs" / "model_qwen_14b_unsloth.json")
print(f"Конфиг модели: {CONFIG_PATH}")

Конфиг модели: /content/drive/MyDrive/VKR/code/configs/model_qwen_14b_unsloth.json


In [19]:
ls  /content/drive/MyDrive/VKR/code/configs

model_qwen_14b_awq.json      model_qwen_3b.json   model_qwen.json
model_qwen_14b_unsloth.json  model_qwen_awq.json


In [20]:
from src.augmentation.stage1_llm_generate import run as run_stage1

run_stage1(CONFIG_PATH)

ЭТАП 1: LLM-генерация (< 15 → 15)
[Данные] Чекпоинт этапа 1 не найден, загружен этап 0: data_after_eda.csv (1755 записей)

[Этап 1] Классов для аугментации: 11
  «Имущественные вопросы»: 2 → нужно ещё 13
  «Подразделение по информационным технологиям»: 2 → нужно ещё 13
  «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: 2 → нужно ещё 13
  «Блок заместителя генерального директора по строительству»: 2 → нужно ещё 13
  «Проект «Обустройство объектов Новейшей нейти»»: 3 → нужно ещё 12
  «Блок исполнительного директора по реализации проекта "Большое месторождение"»: 5 → нужно ещё 10
  «Проект "Обустройство площадных объектов НГКМ Поменбше"»: 6 → нужно ещё 9
  «Управление коммуникаций»: 7 → нужно ещё 8
  «Проект «Обустройство Интересного лицензионного участка»»: 7 → нужно ещё 8
  «Блок бизнес-директора»: 8 → нужно ещё 7
  «Проект "Южный"»: 12 → нужно ещё 3
[LLM] Загружаю модель: unsloth/Qwen2.5-14B-Instruct-bnb-4bit


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Проект "Южный"»: есть 12, нужно ещё 3
  [Попытка 1/10] Нужно ещё 3 текстов
  [Парсинг] Отброшено обрезанное письмо: «Нефтепромышленное предприятие [ORGANIZATION]  
Центр безопас...»
  [Попытка 1] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект "Южный"»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект "Южный"»: прошло 1/1, отсеяно 0
  [Попытка 2/10] Нужно ещё 2 текстов
  [Попытка 2] Распарсено 2 текстов из ответа модели
[Валидация] Класс «Проект "Южный"»: проверяем 2 сгенерированных текстов
[Валидация] Класс «Проект "Южный"»: прошло 2/2, отсеяно 0
[Этап 1] Класс «Проект "Южный"»: добавлено 3 текстов
Пример сгенерированного письма:
--------------------------------------------------
Нефтегазовый концерн [ORGANIZATION]  
Технический отдел  
Адрес: [LOCATION], улица [STREET_NAME], д. [BUILDING_NUMBER], офис [OFFICE_NUMBER]; Тел.: [PHONE_NUMBER]; E-mail: [EMAIL_ADDRESS]; ОКПО [CODE], ОГРН [REGISTRATION_

In [21]:
import torch
print(f"Память GPU: {torch.cuda.memory_allocated() / 1024**3:.1f} ГБ")

Память GPU: 16.0 ГБ


In [22]:
df_after_s1 = load_dataset(stage=1)
dist_s1 = get_class_distribution(df_after_s1)
while (dist_s1 < 15).sum() != 0:# Проверяем результат после этапа 1
    print(f"Записей после этапа 1: {len(df_after_s1)}")
    print(f"Классов с < 15 примерами: {(dist_s1 < 15).sum()}")
    print(f"{"="*100}\n\n")
    print("Повторяем 1й этап")
    print(f"\n\n{"="*100}")
    run_stage1(CONFIG_PATH)
    df_after_s1 = load_dataset(stage=1)
    dist_s1 = get_class_distribution(df_after_s1)
print("Этап 1: Генерация с помощью LLM полностью завершен.")

[Данные] Найден чекпоинт этапа 1: data_after_stage1.csv (1832 записей)
Записей после этапа 1: 1832
Классов с < 15 примерами: 6


Повторяем 1й этап


ЭТАП 1: LLM-генерация (< 15 → 15)
[Данные] Найден чекпоинт этапа 1: data_after_stage1.csv (1832 записей)

[Этап 1] Классов для аугментации: 6
  «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: 2 → нужно ещё 13
  «Блок исполнительного директора по реализации проекта "Большое месторождение"»: 7 → нужно ещё 8
  «Управление коммуникаций»: 10 → нужно ещё 5
  «Подразделение по информационным технологиям»: 11 → нужно ещё 4
  «Проект «Обустройство объектов Новейшей нейти»»: 14 → нужно ещё 1
  «Проект «Обустройство Интересного лицензионного участка»»: 14 → нужно ещё 1
[LLM] Загружаю модель: unsloth/Qwen2.5-14B-Instruct-bnb-4bit


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Проект «Обустройство объектов Новейшей нейти»»: есть 14, нужно ещё 1
  [Попытка 1/10] Нужно ещё 1 текстов
  [Попытка 1] Распарсено 2 текстов из ответа модели
[Валидация] Класс «Проект «Обустройство объектов Новейшей нейти»»: проверяем 2 сгенерированных текстов
[Валидация] Класс «Проект «Обустройство объектов Новейшей нейти»»: прошло 2/2, отсеяно 0
[Этап 1] Класс «Проект «Обустройство объектов Новейшей нейти»»: добавлено 1 текстов
Пример сгенерированного письма:
--------------------------------------------------
[DATE_TIME] исх. No [DOCUMENT_NUMBER] Руководителю проекта [OBJECT_NAME] [ORGANIZATION] [PERSON_FIO]

Уважаемый [PERSON_FIO],

В рамках выполнения договора подряда №[DOCUMENT_NUMBER] на проведение строительно-монтажных работ по объекту [OBJECT_NAME], прошу Вас рассмотреть вопрос о согласовании перечня необходимых строительных материалов и оборудования для продолжения работ на данном этапе.

Наши специалисты подготовили 

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Блок исполнительного директора по реализации проекта "Большое месторождение"»: есть 12, нужно ещё 3
  [Попытка 1/10] Нужно ещё 3 текстов
  [Парсинг] Отброшено обрезанное письмо: «**ООО "Альтернатива", Москва, улица Ленина, дом 15**
Тел.: +...»
  [Попытка 1] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Блок исполнительного директора по реализации проекта "Большое месторождение"»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Блок исполнительного директора по реализации проекта "Большое месторождение"»: прошло 1/1, отсеяно 0
  [Попытка 2/10] Нужно ещё 2 текстов
  [Парсинг] Отброшено обрезанное письмо: «ВРИО генерального директора ПАО [ORGANIZATION] [PERSON],
Зам...»
  [Попытка 2] Распарсено 3 текстов из ответа модели
[Валидация] Класс «Блок исполнительного директора по реализации проекта "Большое месторождение"»: проверяем 3 сгенерированных текстов
  [Длина] Класс «Блок исполнительного директора по реализаци

...

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: есть 4, нужно ещё 11
  [Попытка 1/10] Нужно ещё 11 текстов
  [Парсинг] Отброшено обрезанное письмо: «**О...»
  [Попытка 1] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: прошло 1/1, отсеяно 0
  [Попытка 2/10] Нужно ещё 10 текстов
  [Парсинг] Отброшено обрезанное письмо: «**АО "Нефтетехника"**

Конт...»
  [Попытка 2] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: прошло 1/1, отсеяно 0
  [Попытка 3/10] Нужно ещё 9 текстов
  [Попытка 3] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: про

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: есть 6, нужно ещё 9
  [Попытка 1/10] Нужно ещё 9 текстов
  [Попытка 1] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: прошло 1/1, отсеяно 0
  [Попытка 2/10] Нужно ещё 8 текстов
  [Парсинг] Отброшено обрезанное письмо: «**ООО "ТехноГазстрой"**

Контактная информация:
Адрес: г. Са...»
  [Попытка 2] Распарсено 0 текстов из ответа модели
  [Попытка 3/10] Нужно ещё 8 текстов
  [Парсинг] Отброшено обрезанное письмо: «енные ограничения на проезд техники из-за весеннего паводка ...»
  [Попытка 3] Распарсено 0 текстов из ответа модели
  [Попытка 4/10] Нужно ещё 8 текстов
  [Парсинг] Отброшено обрезанное письмо: «这些例子遵循了您提供的格式，并且是虚构的内容。如果您需要更具体的信息或有其他需求，请告诉我！...»
  [Попытка 4] Распарсено 1 текстов из ответа модели
[Вали

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[LLM] Модель загружена, устройство: cuda:0

[Этап 1] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: есть 13, нужно ещё 2
  [Попытка 1/10] Нужно ещё 2 текстов
  [Парсинг] Отброшено обрезанное письмо: «Уважаемый Петр Сергеевич,

По результатам анализа возможност...»
  [Попытка 1] Распарсено 1 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: проверяем 1 сгенерированных текстов
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: прошло 1/1, отсеяно 0
  [Попытка 2/10] Нужно ещё 1 текстов
  [Парсинг] Отброшено обрезанное письмо: «**ООО "НеваНефть"**

Контактная информация:  
Адрес: г. Санк...»
  [Попытка 2] Распарсено 0 текстов из ответа модели
  [Попытка 3/10] Нужно ещё 1 текстов
  [Попытка 3] Распарсено 2 текстов из ответа модели
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: проверяем 2 сгенерированных текстов
[Валидация] Класс «Проект «Трубопроводный транспорт Ещё одного НГКМ»»: прошло 2

## 3. Этап 2: Парафраз через LLM (< 35 -> 35)

Классы с менее чем 35 примерами пополняются через перефразирование
существующих текстов — LLM получает оригинал и переписывает его
другими словами, сохраняя смысл.

In [ ]:
from src.augmentation.stage2_paraphrase import run as run_stage2

run_stage2(CONFIG_PATH)

In [ ]:
# Проверяем результат после этапа 2
df_after_s2 = load_dataset(stage=2)
dist_s2 = get_class_distribution(df_after_s2)

print(f"Записей после этапа 2: {len(df_after_s2)}")
print(f"Классов с < 35 примерами: {(dist_s2 < 35).sum()}")

## 4. Этап 3: Обратный перевод (< 50 -> 50)

Оставшиеся классы с менее чем 50 примерами дополняются через
обратный перевод RU -> EN -> RU (модели Helsinki-NLP/MarianMT).

In [ ]:
from src.augmentation.stage3_back_translation import run as run_stage3

run_stage3()

In [ ]:
# Проверяем финальный результат
df_final = load_dataset(stage=3)
dist_final = get_class_distribution(df_final)

print(f"Записей после всех этапов: {len(df_final)}")
print(f"Классов с < 50 примерами: {(dist_final < 50).sum()}")
print(f"\nМинимум примеров в классе: {dist_final.min()}")
print(f"Максимум примеров в классе: {dist_final.max()}")

In [ ]:
# Сравниваем распределение до и после аугментации
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

dist.plot(kind="bar", ax=axes[0], color="salmon")
axes[0].set_title("До аугментации")
axes[0].set_ylabel("Количество примеров")
axes[0].tick_params(axis="x", rotation=45)

dist_final.plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("После аугментации")
axes[1].set_ylabel("Количество примеров")
axes[1].axhline(y=50, color="g", linestyle="--", alpha=0.5, label="Целевой минимум (50)")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 5. Классификация — оценка качества аугментации

Запускаем три baseline-модели на финальном датасете.
Все используют BERT-эмбеддинги (ai-forever/sbert_large_nlu_ru)
и StratifiedKFold кросс-валидацию (k=5).

### 5.1 Linear SVM

In [ ]:
from src.classification.run_svm import run as run_svm

run_svm()

### 5.2 Logistic Regression

In [ ]:
from src.classification.run_logreg import run as run_logreg

run_logreg()

### 5.3 Gaussian Naive Bayes

In [ ]:
from src.classification.run_naive_bayes import run as run_naive_bayes

run_naive_bayes()

## 6. Готово

Все этапы пройдены. Результаты:
- Промежуточные датасеты: `Data/data_after_stage{1,2,3}.csv`
- Метрики классификаторов выведены выше

Если нужно перезапустить отдельный этап — удали соответствующий чекпоинт
и запусти ячейку заново.